# Train Main Model Artifacts

## Imports

In [1]:
from pathlib import Path
import sys
import gc
import json
import math
import os
import random
import warnings

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
from IPython.display import display

import joblib

import xgboost as xgb

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "exploration" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from technical.labelling import label_direction_future_avg, label_volatility

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


## Data Loading

In [2]:
SYMBOL = "AAPL"
DATA_DIR = PROJECT_ROOT / "data"
BOOK_PATH = DATA_DIR / f"{SYMBOL}_1s.parquet"
EVENT_PATH = DATA_DIR / f"{SYMBOL}_evt.parquet"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/main_models"

TICK_SIZE = 0.01
HORIZON = 20
SESSION_GAP_SECONDS = 3600
EVENT_FEATURE_LAG_SECONDS = 1
ROLLING_WARMUP = 20
TEST_SESSION_COUNT = 5
EXEC_TYPES = {4, 5}

book = pd.read_parquet(BOOK_PATH).sort_values("time_abs_s").reset_index(drop=True)
book["time_abs_s"] = book["time_abs_s"].astype(np.int64)

event_cols = ["time_abs_s", "event_type", "size", "price", "direction"]
events = pd.read_parquet(
    EVENT_PATH,
    columns=event_cols,
    filters=[("event_type", "in", [1, 2, 3, 4, 5])],
)
events = events.sort_values("time_abs_s").reset_index(drop=True)
events["time_abs_s"] = events["time_abs_s"].astype(np.int64)
events["event_type"] = events["event_type"].astype(np.int8)
events["direction"] = events["direction"].astype(np.int8)
events["size"] = events["size"].astype(np.float32)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame({"object": ["book", "events"], "rows": [len(book), len(events)]}))

,object,rows
0,book,514332
1,events,62337389


## Sessionization

In [3]:
def add_sessions(df):
    out = df.sort_values("time_abs_s").reset_index(drop=True).copy()
    dt = out["time_abs_s"].diff()
    is_new_session = (dt < 0) | (dt > SESSION_GAP_SECONDS)
    out["session_id"] = is_new_session.fillna(False).cumsum().astype(int)
    out["session_pos"] = out.groupby("session_id").cumcount().astype(int)
    out["session_len"] = out.groupby("session_id")["time_abs_s"].transform("size").astype(int)
    out["session_progress"] = (out["session_pos"] / (out["session_len"] - 1)).fillna(0.0).astype(np.float32)
    return out

book = add_sessions(book)

## Event Aggregation

In [4]:
def aggregate_message_features(events):
    evt = events.copy()
    event_type = evt["event_type"].to_numpy(dtype=np.int8)
    direction = evt["direction"].to_numpy(dtype=np.int8)
    size = evt["size"].to_numpy(dtype=np.float32)

    add_mask = event_type == 1
    cancel_mask = np.isin(event_type, [2, 3])
    visible_exec_mask = event_type == 4
    hidden_exec_mask = event_type == 5
    trade_mask = visible_exec_mask | hidden_exec_mask

    evt["limit_add_signed_1s"] = np.where(add_mask, direction * size, 0.0).astype(np.float32)
    evt["limit_add_abs_1s"] = np.where(add_mask, size, 0.0).astype(np.float32)

    evt["cancel_pressure_signed_1s"] = np.where(cancel_mask, -direction * size, 0.0).astype(np.float32)
    evt["cancel_abs_1s"] = np.where(cancel_mask, size, 0.0).astype(np.float32)

    evt["aggr_trade_signed_1s"] = np.where(visible_exec_mask, -direction * size, 0.0).astype(np.float32)
    evt["aggr_trade_abs_1s"] = np.where(visible_exec_mask, size, 0.0).astype(np.float32)

    evt["hidden_trade_signed_1s"] = np.where(hidden_exec_mask, -direction * size, 0.0).astype(np.float32)
    evt["hidden_trade_abs_1s"] = np.where(hidden_exec_mask, size, 0.0).astype(np.float32)

    evt["event_count_1s"] = 1
    evt["trade_count_1s"] = np.where(trade_mask, 1, 0).astype(np.int16)

    message_features = (
        evt.groupby("time_abs_s", sort=True)
        .agg(
            limit_add_signed_1s=("limit_add_signed_1s", "sum"),
            limit_add_abs_1s=("limit_add_abs_1s", "sum"),
            cancel_pressure_signed_1s=("cancel_pressure_signed_1s", "sum"),
            cancel_abs_1s=("cancel_abs_1s", "sum"),
            aggr_trade_signed_1s=("aggr_trade_signed_1s", "sum"),
            aggr_trade_abs_1s=("aggr_trade_abs_1s", "sum"),
            hidden_trade_signed_1s=("hidden_trade_signed_1s", "sum"),
            hidden_trade_abs_1s=("hidden_trade_abs_1s", "sum"),
            event_count_1s=("event_count_1s", "sum"),
            trade_count_1s=("trade_count_1s", "sum"),
        )
        .reset_index()
    )
    message_features["time_abs_s"] = message_features["time_abs_s"] + EVENT_FEATURE_LAG_SECONDS
    return message_features

message_1s = aggregate_message_features(events)
del events
gc.collect()

book = book.merge(message_1s, on="time_abs_s", how="left")
message_cols = [
    "limit_add_signed_1s", "limit_add_abs_1s",
    "cancel_pressure_signed_1s", "cancel_abs_1s",
    "aggr_trade_signed_1s", "aggr_trade_abs_1s",
    "hidden_trade_signed_1s", "hidden_trade_abs_1s",
    "event_count_1s", "trade_count_1s",
]
book[message_cols] = book[message_cols].fillna(0.0)
book["event_count_1s"] = book["event_count_1s"].astype(np.int32)
book["trade_count_1s"] = book["trade_count_1s"].astype(np.int32)
for col in set(message_cols) - {"event_count_1s", "trade_count_1s"}:
    book[col] = book[col].astype(np.float32)

## Selected Features

In [5]:
selected_features = [
    "imb_l1",
    "cum_imb_l3",
    "cum_imb_l5",
    "cum_imb_l10",
    "imbalance_gradient_l3_l10",
    "bid_depth_l3",
    "ask_depth_l3",
    "depth_ratio_l3",
    "spread_ticks",
    "one_tick_spread",
    "touch_depth",
    "cum_depth_l3",
    "cum_depth_l5",
    "cum_depth_l10",
    "microprice",
    "microbias_ticks",
    "microbias_over_spread",
    "event_count_3",
    "event_count_10",
    "event_count_20",
    "trade_count_3",
    "trade_count_10",
    "trade_count_20",
    "ofi_l1_norm_w3",
    "ofi_l1_norm_w10",
    "ofi_l1_norm_w20",
    "ofi_change_3",
    "ofi_change_5",
    "ofi_change_20",
    "mid_ret_3",
    "mid_ret_10",
    "mid_ret_20",
    "realized_vol_5",
    "realized_vol_10",
    "realized_vol_20",
    "return_abs_5",
    "sma_10_dist",
    "sma_20_dist",
    "rsi_14",
    "macd",
    "rolling_return_10",
    "rolling_return_20",
    "bb_position_20",
    "bb_width_20",
    "vwap",
]

## Feature Engineering

In [6]:
def rolling_sum_by_session(series, session_id, window):
    if window == 1:
        return series.astype(np.float32)
    return (
        series.groupby(session_id, sort=False)
        .rolling(window, min_periods=window)
        .sum()
        .reset_index(level=0, drop=True)
        .fillna(0.0)
        .astype(np.float32)
    )


def rolling_mean_by_session(series, session_id, window):
    return (
        series.groupby(session_id, sort=False)
        .rolling(window, min_periods=window)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )


def rolling_std_by_session(series, session_id, window):
    return (
        series.groupby(session_id, sort=False)
        .rolling(window, min_periods=window)
        .std()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )


def ewm_mean_by_session(series, session_id, span):
    return (
        series.groupby(session_id, sort=False)
        .ewm(span=span, adjust=False)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
    )


def compute_ofi_level(df, level):
    grouped = df.groupby("session_id", sort=False)
    bid = df[f"bid{level}"].astype(float)
    ask = df[f"ask{level}"].astype(float)
    bid_size = df[f"bid{level}_sz"].astype(float)
    ask_size = df[f"ask{level}_sz"].astype(float)

    previous_bid = grouped[f"bid{level}"].shift(1)
    previous_ask = grouped[f"ask{level}"].shift(1)
    previous_bid_size = grouped[f"bid{level}_sz"].shift(1).fillna(0.0)
    previous_ask_size = grouped[f"ask{level}_sz"].shift(1).fillna(0.0)

    bid_change = bid - previous_bid
    ask_change = ask - previous_ask
    bid_size_change = bid_size - previous_bid_size
    ask_size_change = ask_size - previous_ask_size

    bid_ofi = np.where(bid_change > 0, bid_size, np.where(bid_change < 0, -previous_bid_size, bid_size_change))
    ask_ofi = np.where(ask_change < 0, -ask_size, np.where(ask_change > 0, previous_ask_size, -ask_size_change))
    return pd.Series(bid_ofi + ask_ofi, index=df.index).fillna(0.0).astype(np.float32)


def build_selected_features(df):
    out = df.copy()
    eps = 1e-9

    book_numeric_cols = [
        "mid",
        *[f"bid{i}" for i in range(1, 11)],
        *[f"ask{i}" for i in range(1, 11)],
        *[f"bid{i}_sz" for i in range(1, 11)],
        *[f"ask{i}_sz" for i in range(1, 11)],
    ]
    out[book_numeric_cols] = out[book_numeric_cols].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    out["mid"] = out["mid"].fillna((out["bid1"] + out["ask1"]) * 0.5).astype(np.float32)

    grouped = out.groupby("session_id", sort=False)

    out["spread"] = (out["ask1"] - out["bid1"]).astype(np.float32)
    out["spread_ticks"] = (out["spread"] / TICK_SIZE).astype(np.float32)
    out["one_tick_spread"] = (out["spread_ticks"] <= 1.01).astype(np.float32)

    out["touch_depth"] = (out["bid1_sz"] + out["ask1_sz"]).astype(np.float32)
    out["imb_l1"] = ((out["bid1_sz"] - out["ask1_sz"]) / (out["touch_depth"] + eps)).astype(np.float32)

    out["bid_depth_l3"] = out[["bid1_sz", "bid2_sz", "bid3_sz"]].sum(axis=1).astype(np.float32)
    out["ask_depth_l3"] = out[["ask1_sz", "ask2_sz", "ask3_sz"]].sum(axis=1).astype(np.float32)
    bid_depth_l5 = out[[f"bid{i}_sz" for i in range(1, 6)]].sum(axis=1).astype(np.float32)
    ask_depth_l5 = out[[f"ask{i}_sz" for i in range(1, 6)]].sum(axis=1).astype(np.float32)
    bid_depth_l10 = out[[f"bid{i}_sz" for i in range(1, 11)]].sum(axis=1).astype(np.float32)
    ask_depth_l10 = out[[f"ask{i}_sz" for i in range(1, 11)]].sum(axis=1).astype(np.float32)

    out["cum_depth_l3"] = (out["bid_depth_l3"] + out["ask_depth_l3"]).astype(np.float32)
    out["cum_depth_l5"] = (bid_depth_l5 + ask_depth_l5).astype(np.float32)
    out["cum_depth_l10"] = (bid_depth_l10 + ask_depth_l10).astype(np.float32)

    out["cum_imb_l3"] = ((out["bid_depth_l3"] - out["ask_depth_l3"]) / (out["cum_depth_l3"] + eps)).astype(np.float32)
    out["cum_imb_l5"] = ((bid_depth_l5 - ask_depth_l5) / (out["cum_depth_l5"] + eps)).astype(np.float32)
    out["cum_imb_l10"] = ((bid_depth_l10 - ask_depth_l10) / (out["cum_depth_l10"] + eps)).astype(np.float32)
    out["imbalance_gradient_l3_l10"] = (out["cum_imb_l3"] - out["cum_imb_l10"]).astype(np.float32)
    out["depth_ratio_l3"] = (out["bid_depth_l3"] / (out["ask_depth_l3"] + eps)).astype(np.float32)

    out["microprice"] = (
        (out["bid1"] * out["ask1_sz"] + out["ask1"] * out["bid1_sz"])
        / (out["touch_depth"] + eps)
    ).astype(np.float32)
    out["microbias_ticks"] = ((out["microprice"] - out["mid"]) / TICK_SIZE).astype(np.float32)
    out["microbias_over_spread"] = (
        (out["microprice"] - out["mid"]) / (out["spread"] + eps)
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    out["event_count_1"] = out["event_count_1s"].astype(np.float32)
    out["trade_count_1"] = out["trade_count_1s"].astype(np.float32)
    for window in [3, 10, 20]:
        out[f"event_count_{window}"] = rolling_sum_by_session(out["event_count_1"], out["session_id"], window)
        out[f"trade_count_{window}"] = rolling_sum_by_session(out["trade_count_1"], out["session_id"], window)

    out["ofi_l1_raw"] = compute_ofi_level(out, 1)
    for window in [3, 10, 20]:
        ofi_sum = rolling_sum_by_session(out["ofi_l1_raw"], out["session_id"], window)
        out[f"ofi_l1_norm_w{window}"] = (
            ofi_sum / (out["touch_depth"] + eps)
        ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    for window in [3, 5, 20]:
        previous_ofi = grouped["ofi_l1_raw"].shift(window).fillna(0.0)
        out[f"ofi_change_{window}"] = (
            (out["ofi_l1_raw"] - previous_ofi) / (out["touch_depth"] + eps)
        ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    log_mid = np.log(out["mid"].astype(float).clip(lower=eps))
    out["mid_ret_1"] = (
        log_mid.groupby(out["session_id"])
        .diff()
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(np.float32)
    )
    for window in [3, 10, 20]:
        out[f"mid_ret_{window}"] = rolling_sum_by_session(out["mid_ret_1"], out["session_id"], window)
        out[f"rolling_return_{window}"] = out[f"mid_ret_{window}"].astype(np.float32)

    out["realized_vol_5"] = np.sqrt(
        rolling_sum_by_session(out["mid_ret_1"].astype(float) ** 2, out["session_id"], 5)
    ).astype(np.float32)
    out["realized_vol_10"] = np.sqrt(
        rolling_sum_by_session(out["mid_ret_1"].astype(float) ** 2, out["session_id"], 10)
    ).astype(np.float32)
    out["realized_vol_20"] = np.sqrt(
        rolling_sum_by_session(out["mid_ret_1"].astype(float) ** 2, out["session_id"], 20)
    ).astype(np.float32)
    out["return_abs_5"] = rolling_sum_by_session(out["mid_ret_1"].abs(), out["session_id"], 5)

    sma_10 = rolling_mean_by_session(out["mid"], out["session_id"], 10)
    sma_20 = rolling_mean_by_session(out["mid"], out["session_id"], 20)
    out["sma_10_dist"] = ((out["mid"] - sma_10) / (sma_10 + eps)).astype(np.float32)
    out["sma_20_dist"] = ((out["mid"] - sma_20) / (sma_20 + eps)).astype(np.float32)

    price_change = grouped["mid"].diff().fillna(0.0)
    gain = price_change.clip(lower=0.0)
    loss = (-price_change).clip(lower=0.0)
    avg_gain = rolling_mean_by_session(gain, out["session_id"], 14)
    avg_loss = rolling_mean_by_session(loss, out["session_id"], 14)
    rs = avg_gain / (avg_loss + eps)
    out["rsi_14"] = (100.0 - (100.0 / (1.0 + rs))).fillna(50.0).astype(np.float32)

    ema_12 = ewm_mean_by_session(out["mid"], out["session_id"], 12)
    ema_26 = ewm_mean_by_session(out["mid"], out["session_id"], 26)
    out["macd"] = (ema_12 - ema_26).astype(np.float32)

    bb_mean_20 = sma_20
    bb_std_20 = rolling_std_by_session(out["mid"], out["session_id"], 20)
    bb_upper_20 = bb_mean_20 + 2.0 * bb_std_20
    bb_lower_20 = bb_mean_20 - 2.0 * bb_std_20
    out["bb_position_20"] = (
        (out["mid"] - bb_lower_20) / (bb_upper_20 - bb_lower_20 + eps)
    ).replace([np.inf, -np.inf], 0.5).fillna(0.5).astype(np.float32)
    out["bb_width_20"] = (
        (bb_upper_20 - bb_lower_20) / (out["mid"].abs() + eps)
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

    trade_volume_1 = (out["aggr_trade_abs_1s"] + out["hidden_trade_abs_1s"]).astype(np.float32)
    vwap_num = rolling_sum_by_session(out["mid"] * trade_volume_1, out["session_id"], 20)
    vwap_den = rolling_sum_by_session(trade_volume_1, out["session_id"], 20)
    out["vwap"] = (vwap_num / vwap_den.replace(0.0, np.nan)).fillna(sma_20).fillna(out["mid"]).astype(np.float32)

    out[selected_features] = (
        out[selected_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(np.float32)
    )
    return out

book = build_selected_features(book)

## Target Construction

In [7]:
def label_one_session(session_df):
    direction_part = label_direction_future_avg(
        session_df,
        price_col="mid",
        horizon=HORIZON,
        out_col="y_dir",
        avg_mode="mean",
        balanced=True,
        verbose=False,
    )[["time_abs_s", "session_id", "y_dir"]]

    volatility_part = label_volatility(
        session_df,
        close_col="mid",
        horizon=HORIZON,
        out_col="y_vol",
    )
    volatility_part = volatility_part[volatility_part["session_pos"] < volatility_part["session_len"] - HORIZON]

    return volatility_part.merge(direction_part, on=["time_abs_s", "session_id"], how="inner")

book = (
    book.groupby("session_id", group_keys=False, sort=False)
    .apply(label_one_session)
    .reset_index(drop=True)
)

book["observed_y_vol"] = book.groupby("session_id", sort=False)["y_vol"].shift(HORIZON)

model_df = book[
    (book["session_pos"] >= ROLLING_WARMUP)
    & book["y_vol"].notna()
    & book["y_dir"].notna()
    & book["observed_y_vol"].notna()
].copy().reset_index(drop=True)

model_df["y_dir"] = model_df["y_dir"].astype(int)
model_df["y_vol"] = model_df["y_vol"].astype(np.float32)

display(model_df[["y_dir", "y_vol"]].describe())

,y_dir,y_vol
count,513452.000000,513452.000000
mean,1.000442,0.000352
std,0.816535,0.000195
min,0.000000,0.000039
25%,0.000000,0.000231
50%,1.000000,0.000305
75%,2.000000,0.000414
max,2.000000,0.003544


## Train/Test Split

In [8]:
sessions = sorted(model_df["session_id"].unique())
test_sessions = sessions[-TEST_SESSION_COUNT:]
train_sessions = sessions[:-TEST_SESSION_COUNT]

model_df["split"] = np.where(model_df["session_id"].isin(test_sessions), "test", "train")

train_df = model_df[model_df["split"] == "train"].copy().reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].copy().reset_index(drop=True)

X_train = train_df[selected_features]
y_dir_train = train_df["y_dir"]
y_vol_train = train_df["y_vol"].to_numpy(dtype=np.float32)

display(pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "sessions": [train_df["session_id"].nunique(), test_df["session_id"].nunique()],
    "min_session": [train_df["session_id"].min(), test_df["session_id"].min()],
    "max_session": [train_df["session_id"].max(), test_df["session_id"].max()],
}))

,split,rows,sessions,min_session,max_session
0,train,396718,17,0,16
1,test,116734,5,17,21


## Direction XGBoost

In [9]:
xgb_direction_params = dict(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=0.0,
    gamma=0.0,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
)

direction_model = xgb.XGBClassifier(**xgb_direction_params)
direction_model.fit(X_train, y_dir_train)

joblib.dump(direction_model, ARTIFACT_DIR / "direction_xgb.joblib")
print("Saved", ARTIFACT_DIR / "direction_xgb.joblib")


Saved /Users/markosmarkides/Documents/quant stuff/Thesis/artifacts/main_models/direction_xgb.joblib


## Volatility XGBoost


In [10]:
xgb_volatility_params = dict(
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    reg_alpha=0.0,
    gamma=0.0,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
)

volatility_model = xgb.XGBRegressor(**xgb_volatility_params)
volatility_model.fit(X_train, y_vol_train)

joblib.dump(volatility_model, ARTIFACT_DIR / "vol_xgb.joblib")

metadata = {
    "symbol": SYMBOL,
    "horizon": HORIZON,
    "rolling_warmup": ROLLING_WARMUP,
    "event_feature_lag_seconds": EVENT_FEATURE_LAG_SECONDS,
    "session_gap_seconds": SESSION_GAP_SECONDS,
    "test_session_count": TEST_SESSION_COUNT,
    "train_sessions": [int(x) for x in train_sessions],
    "test_sessions": [int(x) for x in test_sessions],
    "selected_features": selected_features,
    "direction_model": {
        "model": "XGBClassifier",
        **xgb_direction_params,
    },
    "volatility_model": {
        "model": "XGBRegressor",
        **xgb_volatility_params,
    },
}

(ARTIFACT_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))

print("Saved", ARTIFACT_DIR / "vol_xgb.joblib")
print("Saved", ARTIFACT_DIR / "model_metadata.json")


Saved /Users/markosmarkides/Documents/quant stuff/Thesis/artifacts/main_models/vol_xgb.joblib
Saved /Users/markosmarkides/Documents/quant stuff/Thesis/artifacts/main_models/model_metadata.json
